# 04 - RAG con LangChain y evaluacion con DeepEval

Objetivo: recuperar contexto desde Qdrant, generar una respuesta final con Ollama y evaluar el resultado con metricas de DeepEval.

Este notebook espera que ya hayas ejecutado el notebook de indexacion para tener la coleccion de Qdrant cargada.

In [1]:
import json
import os
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_ollama import ChatOllama, OllamaEmbeddings
from ollama import Client as OllamaClient
from qdrant_client import QdrantClient

from deepeval.metrics import AnswerRelevancyMetric, ContextualRelevancyMetric, FaithfulnessMetric
from deepeval.models.base_model import DeepEvalBaseLLM
from deepeval.test_case import LLMTestCase

os.environ.setdefault("DEEPEVAL_TELEMETRY_OPT_OUT", "YES")
load_dotenv(Path("..").resolve() / ".env")

OLLAMA_PORT = os.getenv("OLLAMA_PORT", "11434")
QDRANT_PORT = os.getenv("QDRANT_PORT", "6333")
CHAT_MODEL = os.getenv("CHAT_MODEL", "llama3")
EMBEDDING_MODEL = os.getenv("EMBEDDING_MODEL", "embeddinggemma")
EVALUATOR_MODEL = os.getenv("EVALUATOR_MODEL", CHAT_MODEL)
COLLECTION_NAME = os.getenv("COLLECTION_NAME", "frasohome_docs")
TOP_K = int(os.getenv("TOP_K", "3"))

OLLAMA_BASE_URL = f"http://127.0.0.1:{OLLAMA_PORT}"
QDRANT_URL = f"http://127.0.0.1:{QDRANT_PORT}"

chat_model = ChatOllama(model=CHAT_MODEL, base_url=OLLAMA_BASE_URL, temperature=0)
embedding_model = OllamaEmbeddings(model=EMBEDDING_MODEL, base_url=OLLAMA_BASE_URL)
client = QdrantClient(url=QDRANT_URL)

print({
    "CHAT_MODEL": CHAT_MODEL,
    "EMBEDDING_MODEL": EMBEDDING_MODEL,
    "EVALUATOR_MODEL": EVALUATOR_MODEL,
    "COLLECTION_NAME": COLLECTION_NAME,
})


{'CHAT_MODEL': 'llama3', 'EMBEDDING_MODEL': 'embeddinggemma', 'EVALUATOR_MODEL': 'llama3', 'COLLECTION_NAME': 'frasohome_docs'}


## Cadena RAG

La funcion `ask_rag` devuelve tres elementos: respuesta, hits recuperados y contexto construido para el prompt.

In [2]:
prompt = ChatPromptTemplate.from_template(
    """
Eres un asistente de soporte de FrasoHome.
Responde solo con la informacion del contexto.
Si el contexto no es suficiente, di claramente que no lo sabes.

Pregunta: {question}

Contexto:
{context}
    """.strip()
)
chain = prompt | chat_model | StrOutputParser()

def retrieve(question: str, top_k: int = TOP_K):
    query_vector = embedding_model.embed_query(question)
    return client.query_points(
        collection_name=COLLECTION_NAME,
        query=query_vector,
        limit=top_k,
    ).points

def build_context(hits) -> str:
    blocks = []
    for hit in hits:
        source = hit.payload.get("source", "desconocido")
        chunk_id = hit.payload.get("chunk_id", "?")
        text = hit.payload.get("text", "")
        blocks.append(f"[source={source} chunk={chunk_id}]\n{text}")
    return "\n\n".join(blocks)

def ask_rag(question: str, top_k: int = TOP_K):
    hits = retrieve(question, top_k=top_k)
    context = build_context(hits)
    answer = chain.invoke({"question": question, "context": context})
    return answer, hits, context


In [3]:
question = "Puedo devolver una mesa ya montada si solo he cambiado de opinion?"
answer, hits, context = ask_rag(question)

print("RESPUESTA")
print(answer)
print("\nFUENTES")
for hit in hits:
    print({
        "score": round(hit.score, 4),
        "source": Path(hit.payload.get("source", "")).name,
        "chunk_id": hit.payload.get("chunk_id"),
    })


RESPUESTA
No, no puedes devolver una mesa ya montada si solo has cambiado de opinión. Según nuestra política de devoluciones, los productos ya montados no se aceptan como devolución por desistimiento si el motivo es simplemente un cambio de opinión, color o medida percibida por el cliente tras el montaje.

FUENTES
{'score': 0.6364, 'source': 'politica_devoluciones.md', 'chunk_id': 5}
{'score': 0.5839, 'source': 'politica_devoluciones.md', 'chunk_id': 4}
{'score': 0.5227, 'source': 'politica_devoluciones.md', 'chunk_id': 7}


## Modelo evaluador para DeepEval

DeepEval puede usar OpenAI por defecto. Para mantener el laboratorio local, este adaptador usa Ollama como juez. Si quieres separar el modelo que responde del modelo que evalua, define `EVALUATOR_MODEL` en `.env`.

In [4]:
class OllamaDeepEvalModel(DeepEvalBaseLLM):
    def __init__(self, model_name: str, host: str, temperature: float = 0):
        self.model_name = model_name
        self.host = host
        self.temperature = temperature
        self._client = OllamaClient(host=host)
        super().__init__(model=model_name)

    def load_model(self):
        return self._client

    def get_model_name(self) -> str:
        return f"ollama/{self.model_name}"

    def supports_json_mode(self):
        return True

    def supports_structured_outputs(self):
        return True

    def _generate_content(self, prompt: str, schema=None):
        format_arg = None
        if schema is not None and hasattr(schema, "model_json_schema"):
            format_arg = schema.model_json_schema()
        elif schema is not None:
            format_arg = "json"

        response = self.model.chat(
            model=self.model_name,
            messages=[
                {
                    "role": "system",
                    "content": "Eres un evaluador estricto. Si se solicita JSON, responde solo JSON valido.",
                },
                {"role": "user", "content": prompt},
            ],
            options={"temperature": self.temperature},
            format=format_arg,
        )
        content = response["message"]["content"]

        if schema is None:
            return content

        try:
            return schema.model_validate_json(content)
        except Exception:
            return schema.model_validate(json.loads(content))

    def generate(self, prompt: str, *args, schema=None, **kwargs):
        return self._generate_content(prompt, schema=schema)

    async def a_generate(self, prompt: str, *args, schema=None, **kwargs):
        return self._generate_content(prompt, schema=schema)

evaluator_model = OllamaDeepEvalModel(
    model_name=EVALUATOR_MODEL,
    host=OLLAMA_BASE_URL,
)
print(evaluator_model.get_model_name())


ollama/llama3


## Generar casos de evaluacion

El CSV aporta preguntas, fuente esperada y si el sistema deberia abstenerse. Las metricas de DeepEval evaluan respuesta y contexto; los checks deterministas revisan trazabilidad y abstencion.

In [5]:
evaluation_path = Path("..").resolve() / "evaluation" / "questions.csv"
eval_df = pd.read_csv(evaluation_path)

def source_names(hits) -> list[str]:
    return [Path(hit.payload.get("source", "")).name for hit in hits]

def looks_like_abstention(text: str) -> bool:
    text = text.lower()
    markers = [
        "no lo se",
        "no tengo suficiente",
        "no dispongo",
        "no aparece",
        "no se menciona",
        "contexto no es suficiente",
    ]
    return any(marker in text for marker in markers)

test_cases = []
case_rows = []

for row in eval_df.itertuples(index=False):
    answer, hits, context = ask_rag(row.question)
    retrieval_context = [hit.payload.get("text", "") for hit in hits]
    expected_source = row.expected_source if isinstance(row.expected_source, str) else ""
    sources = source_names(hits)

    test_case = LLMTestCase(
        input=row.question,
        actual_output=answer,
        retrieval_context=retrieval_context,
        expected_output=row.notes,
        name=row.id,
        additional_metadata={
            "expected_source": expected_source,
            "retrieved_sources": sources,
            "should_abstain": bool(row.should_abstain),
        },
    )
    test_cases.append(test_case)
    case_rows.append({
        "id": row.id,
        "question": row.question,
        "answer": answer,
        "expected_source": expected_source,
        "top_source": sources[0] if sources else None,
        "source_match": (expected_source in sources) if expected_source else None,
        "should_abstain": bool(row.should_abstain),
        "abstention_match": looks_like_abstention(answer) == bool(row.should_abstain),
    })

cases_df = pd.DataFrame(case_rows)
cases_df[["id", "question", "expected_source", "top_source", "source_match", "should_abstain", "abstention_match"]]


,id,question,expected_source,top_source,source_match,should_abstain,abstention_match
0,Q1,¿Puedo devolver una mesa ya montada si simplem...,politica_devoluciones.md,politica_devoluciones.md,True,False,True
1,Q2,¿Qué herramientas necesito para montar la mesa...,manual_mesa_oslo.md,manual_mesa_oslo.md,True,False,True
2,Q3,¿Cuánto tiempo tengo para comunicar un daño de...,politica_devoluciones.md,politica_devoluciones.md,True,False,True
3,Q4,¿Puedo devolver un producto personalizado?,politica_devoluciones.md,politica_devoluciones.md,True,False,True
4,Q5,¿Qué hago si faltan tornillos en la caja?,manual_mesa_oslo.md,manual_mesa_oslo.md,True,False,True
5,Q6,¿Cuál es el plazo de entrega de la mesa Oslo?,,manual_mesa_oslo.md,None,True,False
6,Q7,¿FrasoHome ofrece servicio de montaje a domici...,,politica_devoluciones.md,None,True,False
7,Q8,¿Debo apretar totalmente los tornillos al prin...,manual_mesa_oslo.md,manual_mesa_oslo.md,True,False,True


## Ejecutar metricas de DeepEval

- `AnswerRelevancyMetric`: si la respuesta responde a la pregunta.
- `FaithfulnessMetric`: si la respuesta se sostiene en el contexto recuperado.
- `ContextualRelevancyMetric`: si el contexto recuperado es relevante para la pregunta.

In [6]:
metrics = [
    AnswerRelevancyMetric(threshold=0.7, model=evaluator_model, async_mode=False),
    FaithfulnessMetric(threshold=0.7, model=evaluator_model, async_mode=False),
    ContextualRelevancyMetric(threshold=0.7, model=evaluator_model, async_mode=False),
]

metric_rows = []
for test_case in test_cases:
    row = {"id": test_case.name}
    for metric in metrics:
        metric_name = metric.__class__.__name__.replace("Metric", "")
        try:
            score = metric.measure(
                test_case,
                _show_indicator=False,
                _log_metric_to_confident=False,
            )
            row[f"{metric_name}_score"] = round(score, 3)
            row[f"{metric_name}_success"] = metric.success
            row[f"{metric_name}_reason"] = metric.reason
        except Exception as exc:
            row[f"{metric_name}_score"] = None
            row[f"{metric_name}_success"] = False
            row[f"{metric_name}_reason"] = f"ERROR: {exc}"
    metric_rows.append(row)

metrics_df = pd.DataFrame(metric_rows)
metrics_df


,id,AnswerRelevancy_score,AnswerRelevancy_success,AnswerRelevancy_reason,Faithfulness_score,Faithfulness_success,Faithfulness_reason,ContextualRelevancy_score,ContextualRelevancy_success,ContextualRelevancy_reason
0,Q1,0.0,False,The score is 0.00 because the actual output do...,0.00,False,The score is 0.00 because the actual output cl...,0.667,False,The score is 0.67 because the input question a...
1,Q2,1.0,True,The score is 1.00 because the answer directly ...,0.75,True,The score is 0.75 because the actual output me...,0.600,False,The score is 0.60 because the retrieval contex...
2,Q3,1.0,True,The score is 1.00 because the answer directly ...,1.00,True,The score is 1.00 because there are no contrad...,0.571,False,The score is 0.57 because the retrieval contex...
3,Q4,0.0,False,The score is 0.00 because You cannot return a ...,0.50,False,The score is 0.50 because the actual output co...,0.667,False,The score is 0.67 because the retrieval contex...
4,Q5,1.0,True,The score is 1.00 because the answer directly ...,0.75,True,The score is 0.75 because The context states '...,0.250,False,The score is 0.25 because the retrieval contex...
5,Q6,0.5,False,The score is 0.50 because the answer does not ...,0.50,False,The score is 0.50 because the actual output do...,0.727,True,The score is 0.73 because the retrieval contex...
6,Q7,0.0,False,The score is 0.00 because The statement explic...,0.00,False,The score is 0.00 because the actual output co...,0.833,True,The score is 0.83 because the statement about ...
7,Q8,0.5,False,The score is 0.50 because the answer only part...,1.00,True,The score is 1.00 because there are no contrad...,0.286,False,The score is 0.29 because the retrieval contex...


In [7]:
report_df = cases_df.merge(metrics_df, on="id", how="left")

score_columns = [column for column in report_df.columns if column.endswith("_score")]
summary = {
    "source_accuracy": report_df["source_match"].dropna().mean(),
    "abstention_accuracy": report_df["abstention_match"].mean(),
}
for column in score_columns:
    summary[column.replace("_score", "_avg")] = report_df[column].mean()

print(summary)
report_df[[
    "id",
    "source_match",
    "abstention_match",
    *score_columns,
]]


{'source_accuracy': np.float64(1.0), 'abstention_accuracy': np.float64(0.75), 'AnswerRelevancy_avg': np.float64(0.5), 'Faithfulness_avg': np.float64(0.5625), 'ContextualRelevancy_avg': np.float64(0.575125)}


,id,source_match,abstention_match,AnswerRelevancy_score,Faithfulness_score,ContextualRelevancy_score
0,Q1,True,True,0.0,0.00,0.667
1,Q2,True,True,1.0,0.75,0.600
2,Q3,True,True,1.0,1.00,0.571
3,Q4,True,True,0.0,0.50,0.667
4,Q5,True,True,1.0,0.75,0.250
5,Q6,None,False,0.5,0.50,0.727
6,Q7,None,False,0.0,0.00,0.833
7,Q8,True,True,0.5,1.00,0.286
